# Runnable Fallbacks

The `fallbacks.py` module defines a serializable Runnable wrapper that executes a primary Runnable and uses alternative Runnables when handled failures occur.

Fallbacks are attempted in order until one succeeds or every Runnable fails. They can be applied to an individual Runnable or to an entire Runnable chain.

# RunnableWithFallbacks

`RunnableWithFallbacks` wraps a primary Runnable and an ordered sequence of fallback Runnables.

Only exceptions listed in `exceptions_to_handle` activate the fallback sequence. Other exceptions are raised immediately. The first handled exception is raised when every Runnable fails.

## Bases

- `RunnableSerializable[Input, Output]`

## Attributes

1. `runnable`: Stores the primary Runnable that is executed first.
   * **Type:**
     ```python
     runnable: Runnable[Input, Output]
     ```

2. `fallbacks`: Stores the fallback Runnables in the order in which they should be attempted.
   * **Type:**
     ```python
     fallbacks: Sequence[
         Runnable[Input, Output]
     ]
     ```

3. `exceptions_to_handle`: Stores the exception types that activate fallback execution.

   An exception that is not an instance of one of these types is raised immediately.

   * **Type:**
     ```python
     exceptions_to_handle: tuple[
         type[BaseException],
         ...
     ] = (Exception,)
     ```

4. `exception_key`: Stores the dictionary key through which the most recent handled exception is passed to the next fallback.

   When this field is not `None`, every input must be a dictionary. When it is `None`, handled exceptions are not added to the fallback input.

   * **Type:**
     ```python
     exception_key: str | None = None
     ```

## Configuration

1. `model_config`: Allows arbitrary Python types in the Pydantic model.
   * **Definition:**
     ```python
     model_config = ConfigDict(
         arbitrary_types_allowed=True
     )
     ```

### Properties

1. `InputType`: Returns the input type accepted by the primary Runnable.
   * **Type:**
     ```python
     InputType: type[Input]
     ```

2. `OutputType`: Returns the output type produced by the primary Runnable.
   * **Type:**
     ```python
     OutputType: type[Output]
     ```

3. `config_specs`: Returns the unique configuration specifications exposed by the primary Runnable and every fallback Runnable.
   * **Type:**
     ```python
     config_specs: list[
         ConfigurableFieldSpec
     ]
     ```

4. `runnables`: Returns an iterator that yields the primary Runnable followed by each fallback Runnable.
   * **Type:**
     ```python
     runnables: Iterator[
         Runnable[Input, Output]
     ]
     ```

### Methods

1. `get_input_schema`: Returns the input schema of the primary Runnable.
   * **Syntax:**
     ```python
     get_input_schema(
         self,
         config: RunnableConfig | None = None # Configuration used to generate the schema
     ) -> TypeBaseModel
     ```

2. `get_output_schema`: Returns the output schema of the primary Runnable.
   * **Syntax:**
     ```python
     get_output_schema(
         self,
         config: RunnableConfig | None = None # Configuration used to generate the schema
     ) -> TypeBaseModel
     ```

3. `is_lc_serializable`: Indicates that `RunnableWithFallbacks` supports LangChain serialization.
   * **Syntax:**
     ```python
     @classmethod
     is_lc_serializable(
         cls
     ) -> bool
     ```

4. `get_lc_namespace`: Returns the LangChain serialization namespace.
   * **Syntax:**
     ```python
     @classmethod
     get_lc_namespace(
         cls
     ) -> list[str]
     ```

5. `invoke`: Synchronously invokes the primary Runnable and then tries each fallback when a handled exception occurs.

   The most recent handled exception is inserted into the input under `exception_key` when that field is configured. A root chain callback is created, and every attempted Runnable receives a child callback configuration.

   If every Runnable fails with handled exceptions, the first handled exception is raised.

   * **Syntax:**
     ```python
     invoke(
         self,
         input: Input, # Input passed to the primary Runnable and fallbacks
         config: RunnableConfig | None = None, # Runtime configuration
         **kwargs: Any # Additional invocation arguments
     ) -> Output
     ```

6. `ainvoke`: Asynchronously invokes the primary Runnable and then tries each fallback when a handled exception occurs.

   It follows the same fallback, exception-injection, and callback rules as `invoke`.

   * **Syntax:**
     ```python
     async ainvoke(
         self,
         input: Input, # Input passed to the primary Runnable and fallbacks
         config: RunnableConfig | None = None, # Runtime configuration
         **kwargs: Any | None # Additional invocation arguments
     ) -> Output
     ```

7. `batch`: Synchronously processes multiple inputs using the primary Runnable and fallback sequence.

   Successful inputs are removed from subsequent fallback attempts. Only inputs that produced handled exceptions are retried by the next Runnable.

   Unhandled exceptions are raised immediately unless `return_exceptions=True`. When every fallback fails for an input, its handled exception is returned or raised according to `return_exceptions`.

   * **Syntax:**
     ```python
     batch(
         self,
         inputs: list[Input], # Inputs to process
         config: RunnableConfig
         | list[RunnableConfig]
         | None = None, # Configuration for one or more inputs
         *,
         return_exceptions: bool = False, # Return exceptions instead of raising them
         **kwargs: Any | None # Additional invocation arguments
     ) -> list[Output]
     ```

8. `abatch`: Asynchronously processes multiple inputs using the primary Runnable and fallback sequence.

   Successful inputs are removed from subsequent attempts, while only inputs that produced handled exceptions are passed to the next fallback.

   It follows the same exception-return and callback rules as `batch`.

   * **Syntax:**
     ```python
     async abatch(
         self,
         inputs: list[Input], # Inputs to process
         config: RunnableConfig
         | list[RunnableConfig]
         | None = None, # Configuration for one or more inputs
         *,
         return_exceptions: bool = False, # Return exceptions instead of raising them
         **kwargs: Any | None # Additional invocation arguments
     ) -> list[Output]
     ```

9. `stream`: Synchronously streams output from the first Runnable that successfully produces an initial chunk.

   A handled exception raised before the first chunk causes the next fallback to be attempted. Once a Runnable begins streaming successfully, later streaming errors are raised without trying another fallback.

   Streamed chunks are combined when their output type supports addition, and the combined output is reported to the callback manager.

   * **Syntax:**
     ```python
     stream(
         self,
         input: Input, # Input passed to the primary Runnable and fallbacks
         config: RunnableConfig | None = None, # Runtime configuration
         **kwargs: Any | None # Additional streaming arguments
     ) -> Iterator[Output]
     ```

10. `astream`: Asynchronously streams output from the first Runnable that successfully produces an initial chunk.

    A handled exception raised before the first chunk causes the next fallback to be attempted. Once asynchronous streaming begins successfully, later errors are raised without trying another fallback.

    * **Syntax:**
      ```python
      async astream(
          self,
          input: Input, # Input passed to the primary Runnable and fallbacks
          config: RunnableConfig | None = None, # Runtime configuration
          **kwargs: Any | None # Additional streaming arguments
      ) -> AsyncIterator[Output]
      ```

11. `__getattr__`: Delegates missing attributes to the primary Runnable.

    Non-callable attributes and methods that do not return a Runnable are taken directly from the primary Runnable.

    When a delegated method returns a Runnable, that method is applied to both the primary Runnable and every fallback, and a new `RunnableWithFallbacks` wrapper is returned.

    * **Syntax:**
      ```python
      __getattr__(
          self,
          name: str # Name of the delegated attribute or method
      ) -> Any
      ```